# PT-08 — GRPO from scratch : signal de récompense dans un environnement vérifié par Z3

**Navigation** : [Version Python](PT_08_grpo_from_scratch_toy_env.ipynb) · [PT-07](PT_07_rewardspy_reward_hacking.ipynb) · [PostTraining](README.md) · [PT-09](PT_09_rloo_from_scratch_toy_env.ipynb)

**Objectif** : exécuter et comparer REINFORCE, PPO et GRPO sur un bandit quadratique, en vérifiant la récompense avec Z3. **Prérequis** : .NET Interactive, TorchSharp-cpu 0.107.0 et le binaire Z3 4.12.2 ; préparation portable dans le [README](README.md). **Durée indicative** : 5 minutes CPU. Aucun GPU ni clé.

TorchSharp et libtorch calculent réellement les réseaux, gradients et mises à jour ; Z3 contrôle la formule. GRPO est ici un objectif pédagogique sans critic, modèle de référence, KL ni séquence de tokens. Les graines et générateurs C# diffèrent de Python.

Les éventuels avertissements CS1701 indiquent une différence de versions de références entre TorchSharp, .NET Interactive et le runtime ; ils restent visibles dans les sorties.


In [1]:
#r "nuget: TorchSharp-cpu, 0.107.0"
#r "nuget: Microsoft.Z3, 4.12.2"
using System;
using System.Collections.Generic;
using System.Linq;
using System.Diagnostics;
using System.Globalization;
using System.Text;
using System.IO;
using System.Runtime.InteropServices;
using TorchSharp;
using static TorchSharp.torch;
using Microsoft.DotNet.Interactive.Formatting;
using Z3 = Microsoft.Z3;
torch.set_num_threads(1);
torch.manual_seed(42);
string z3Library=Environment.GetEnvironmentVariable("Z3_NATIVE_LIBRARY");
if(string.IsNullOrWhiteSpace(z3Library)||!File.Exists(z3Library))throw new Exception("Configurer Z3_NATIVE_LIBRARY avant le kernel : voir README.");
NativeLibrary.SetDllImportResolver(typeof(Z3.Context).Assembly,(name,assembly,path)=>name=="libz3"?NativeLibrary.Load(z3Library):IntPtr.Zero);
Console.WriteLine($".NET {Environment.Version} ; TorchSharp-cpu 0.107.0 ; libtorch CPU 2.10.0 ; Z3 {Z3.Version.FullVersion} ; torch threads={torch.get_num_threads()} ; seed=42");

Installed Packages Microsoft.Z3, 4.12.2 TorchSharp-cpu, 0.107.0

Loading extensions from `/workspace/coursia-toolchains/nuget/skiasharp/2.88.6/interactive-extensions/dotnet/SkiaSharp.DotNet.Interactive.dll`

.NET 10.0.14 ; TorchSharp-cpu 0.107.0 ; libtorch CPU 2.10.0 ; Z3 Z3 4.12.2.0 ; torch threads=1 ; seed=42


## 1. Le fil rouge : baseline du critic ou du groupe

PPO apprend une valeur $V(s)$ ; ce GRPO soustrait la moyenne empirique des récompenses des $G$ actions d'un même prompt et divise par leur écart-type. Cette moyenne inclut la récompense courante et n'est pas une valeur exacte connue. Un groupe constant ne fournit pas de préférence.

## 2. Environnement quadratique et vérification Z3

Pour $a\in[0,5]$, $b=t^2+a$ et $t\in[0,15]$, l'action est une racine candidate dans $[0,15]$. L'entraînement utilise une égalité tensorielle rapide ; le solveur SMT vérifie séparément son équivalence, pas chaque échantillon pendant l'entraînement.

In [2]:
const int Actions=16,ADim=6,BDim=231,InputDim=237;
var ctx=new Z3.Context();
var solver=ctx.MkSolver();
var zAction=ctx.MkIntConst("action");
bool Verify(int a,int b,int action)
{
    solver.Push();
    solver.Add(ctx.MkEq(zAction,ctx.MkInt(action)),ctx.MkGe(zAction,ctx.MkInt(0)),
        ctx.MkLt(zAction,ctx.MkInt(Actions)),
        ctx.MkEq(ctx.MkAdd(ctx.MkMul(zAction,zAction),ctx.MkInt(a)),ctx.MkInt(b)));
    bool result=solver.Check()==Z3.Status.SATISFIABLE;
    solver.Pop();
    return result;
}
bool Fast(int a,int b,int action)=>action>=0 && action<Actions && action*action+a==b;
foreach(var (a,b) in new[]{(2,2),(3,7),(4,173),(5,230)})
{
    var solutions=Enumerable.Range(0,Actions).Where(x=>Verify(a,b,x)).ToArray();
    Console.WriteLine($"(a={a}, b={b}) : solutions Z3 [{string.Join(",",solutions)}]");
}
Tensor Reward(Tensor a,Tensor b,Tensor action)
{
    return ((action*action+a).eq(b).logical_and(action.ge(0)).logical_and(action.lt(Actions))).to_type(ScalarType.Float32);
}

var rngCheck=new Random(42);
var allA=new List<long>();var allB=new List<long>();var allActions=new List<long>();var z3Expected=new List<float>();
int checks=0,mismatches=0;
for(int i=0;i<2400;i++)
{
    int a,target,action;
    if(i<1536){a=i/256;target=(i/16)%16;action=i%16;}
    else{a=rngCheck.Next(6);target=rngCheck.Next(16);action=rngCheck.Next(-4,20);}
    int b=target*target+a;
    bool z3Result=Verify(a,b,action);
    if(z3Result!=Fast(a,b,action))mismatches++;
    allA.Add(a);allB.Add(b);allActions.Add(action);z3Expected.Add(z3Result?1f:0f);
    checks++;
}
using(var proofScope=torch.NewDisposeScope())
{
    var at=torch.tensor(allA.ToArray());var bt=torch.tensor(allB.ToArray());var acts=torch.tensor(allActions.ToArray());
    var tensorResult=Reward(at,bt,acts).data<float>().ToArray();
    mismatches+=tensorResult.Zip(z3Expected).Count(x=>x.First!=x.Second);
}
if(mismatches!=0 || !Verify(2,6,2) || Verify(2,6,3) || Verify(0,0,-1) || Verify(0,0,16))throw new Exception("Divergence Z3 / fast-path");
var randomBase=new Random(0);int correct=0;
for(int i=0;i<4000;i++){int a=randomBase.Next(6),t=randomBase.Next(16),act=randomBase.Next(16);if(Fast(a,t*t+a,act))correct++;}
Console.WriteLine($"SMT / récompense tensorielle : {checks} vérifications, {mismatches} divergence ; hasard : {correct}/4000 = {correct/4000.0:F3}, attendu 1/16.");

(a=2, b=2) : solutions Z3 [0]


(a=3, b=7) : solutions Z3 [2]


(a=4, b=173) : solutions Z3 [13]


(a=5, b=230) : solutions Z3 [15]


SMT / récompense tensorielle : 2400 vérifications, 0 divergence ; hasard : 233/4000 = 0.058, attendu 1/16.



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply r

### Lecture du résultat

Les quatre énumérations donnent les racines valides. Les 1536 triplets du domaine et 864 cas supplémentaires, y compris des actions hors bornes, comparent Z3 à la fonction de récompense tensorielle utilisée en entraînement. Le hasard est proche de 1/16 ; 96 prompts one-hot sans jeu séparé ne prouvent pas une généralisation au calcul de racine.

## 3. Acteur et critic

Deux réseaux de largeur cachée 64 ont des paramètres distincts : acteur 237→64→16, critic 237→64→1. PPO utilise les deux ; REINFORCE et GRPO utilisent l'acteur seul.

In [3]:
torch.nn.Module<Tensor,Tensor> Actor()=>nn.Sequential(nn.Linear(InputDim,64),nn.ReLU(),nn.Linear(64,Actions));
torch.nn.Module<Tensor,Tensor> Critic()=>nn.Sequential(nn.Linear(InputDim,64),nn.ReLU(),nn.Linear(64,1));
long Params(torch.nn.Module<Tensor,Tensor> m)=>m.parameters().Sum(p=>p.numel());
var actorCount=Actor();
var criticCount=Critic();
if(Params(actorCount)!=16272||Params(criticCount)!=15297)throw new Exception("Architecture incorrecte");
Console.WriteLine($"Acteur : {Params(actorCount)} paramètres ; critic : {Params(criticCount)} ; PPO : {Params(actorCount)+Params(criticCount)} ; GRPO : {Params(actorCount)}.");

Acteur : 16272 paramètres ; critic : 15297 ; PPO : 31569 ; GRPO : 16272.


### Lecture du résultat

Le critic représente 15 297 des 31 569 paramètres des deux réseaux dans ce toy. Cette proportion ne prédit pas la mémoire totale d'un entraînement LLM.

## 4. REINFORCE : moyenne du batch

La politique est mise à jour à partir de l'avantage `reward − moyenne du batch` : 400 étapes, 32 actions par étape, Adam au taux 0,005.

In [4]:
(long[] a,long[] b) Batch(Random rng,int n,int group=1)
{
    var a=new long[n*group];var b=new long[n*group];
    for(int i=0;i<n;i++)
    {
        int aa=rng.Next(ADim),target=rng.Next(Actions);
        for(int j=0;j<group;j++){a[i*group+j]=aa;b[i*group+j]=target*target+aa;}
    }
    return (a,b);
}
Tensor Encode(Tensor a,Tensor b)
{
    var aa=nn.functional.one_hot(a,ADim).to_type(ScalarType.Float32);
    var bb=nn.functional.one_hot(b,BDim).to_type(ScalarType.Float32);
    return torch.cat(new[]{aa,bb},dim:1);
}
(Tensor action,Tensor logp) Sample(torch.nn.Module<Tensor,Tensor> actor,Tensor state)
{
    var logProbs=actor.forward(state).log_softmax(1);
    var actions=logProbs.exp().multinomial(1).squeeze(1);
    var chosen=logProbs.gather(1,actions.unsqueeze(1)).squeeze(1);
    return (actions,chosen);
}
(List<double> history,torch.nn.Module<Tensor,Tensor> policy) TrainReinforce(int steps=400,int batch=32,int seed=42)
{
    torch.manual_seed(seed);var rng=new Random(seed);
    var actor=Actor();using var optimizer=torch.optim.Adam(actor.parameters(),lr:0.005);
    var history=new List<double>();
    for(int step=0;step<steps;step++)
    {
        using var scope=torch.NewDisposeScope();
        var (a,b)=Batch(rng,batch);var at=torch.tensor(a);var bt=torch.tensor(b);var state=Encode(at,bt);
        var (action,logp)=Sample(actor,state);var r=Reward(at,bt,action);var advantage=r-r.mean();
        var loss=-(logp*advantage).mean();
        optimizer.zero_grad();loss.backward();optimizer.step();history.Add(r.mean().item<float>());
    }
    return (history,actor);
}
var reinforceResult=TrainReinforce();var hReinforce=reinforceResult.history;
Console.WriteLine($"REINFORCE : {hReinforce.Count} étapes, première récompense={hReinforce.First():F3}, dernière={hReinforce.Last():F3}.");

REINFORCE : 400 étapes, première récompense=0.000, dernière=0.656.



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



### Lecture du résultat

La courbe provient de 400 mises à jour réelles. Les valeurs initiale et finale sont des moyennes de batches tirés et ne sont pas une évaluation hors échantillon.

## 5. PPO : clipping et critic appris

Pour chaque batch de 64 actions, l'ancienne politique, les actions et l'avantage `reward − critic` sont figés ; quatre époques optimisent l'objectif clippé, la MSE du critic et l'entropie.

In [5]:
Tensor ClippedLoss(Tensor ratio,Tensor advantage,double clip=0.2)
{
    return -torch.minimum(ratio*advantage,ratio.clamp(1-clip,1+clip)*advantage).mean();
}
using(var testScope=torch.NewDisposeScope())
{
    var ratios=torch.tensor(new float[]{1.5f,0.5f});var adv=torch.tensor(new float[]{1,-1});
    double clipped=ClippedLoss(ratios,adv).item<float>();double raw=-(ratios*adv).mean().item<float>();
    if(Math.Abs(clipped+0.2)>1e-6||Math.Abs(raw+0.5)>1e-6)throw new Exception("Fixture clipping incorrecte");
    using var testActor=nn.Linear(2,2);using var opt=torch.optim.Adam(testActor.parameters(),lr:0.005);
    var before=testActor.weight.detach().clone();var x=torch.tensor(new float[]{1,0,0,1}).reshape(2,2);
    var loss=testActor.forward(x).square().mean();opt.zero_grad();loss.backward();
    double gradient=testActor.weight.grad.abs().sum().item<float>();opt.step();
    double delta=(testActor.weight-before).abs().sum().item<float>();
    if(!(gradient>0&&delta>0))throw new Exception("Gradient ou mise à jour nul");
    Console.WriteLine($"Fixture clipping : {clipped:F3} contre {raw:F3} sans clipping ; gradient L1={gradient:F5}, variation poids L1={delta:F5}.");
}
(List<double> history,torch.nn.Module<Tensor,Tensor> policy,torch.nn.Module<Tensor,Tensor> critic) TrainPpo(int steps=400,int batch=64,int seed=42)
{
    torch.manual_seed(seed);var rng=new Random(seed);
    var actor=Actor();var critic=Critic();
    using var optimizer=torch.optim.Adam(actor.parameters().Concat(critic.parameters()),lr:0.005);
    var history=new List<double>();
    for(int step=0;step<steps;step++)
    {
        using var scope=torch.NewDisposeScope();
        var (a,b)=Batch(rng,batch);var at=torch.tensor(a);var bt=torch.tensor(b);var state=Encode(at,bt);
        Tensor action,oldLogp,r,advantage;
        using(torch.no_grad())
        {
            (action,oldLogp)=Sample(actor,state);r=Reward(at,bt,action);
            advantage=r-critic.forward(state).squeeze(1);
        }
        for(int epoch=0;epoch<4;epoch++)
        {
            using var epochScope=torch.NewDisposeScope();
            var logProbs=actor.forward(state).log_softmax(1);
            var logp=logProbs.gather(1,action.unsqueeze(1)).squeeze(1);
            var ratio=(logp-oldLogp).exp();
            var policyLoss=ClippedLoss(ratio,advantage);
            var valueLoss=(critic.forward(state).squeeze(1)-r).square().mean();
            var entropy=-(logProbs.exp()*logProbs).sum(1).mean();
            var loss=policyLoss+0.5*valueLoss-0.01*entropy;
            optimizer.zero_grad();loss.backward();optimizer.step();
        }
        history.Add(r.mean().item<float>());
    }
    return (history,actor,critic);
}
var ppoResult=TrainPpo();var hPpo=ppoResult.history;
Console.WriteLine($"PPO : {hPpo.Count} étapes, première récompense={hPpo.First():F3}, dernière={hPpo.Last():F3}, 4 époques.");

Fixture clipping : -0.200 contre -0.500 sans clipping ; gradient L1=0.43805, variation poids L1=0.02000.


PPO : 400 étapes, première récompense=0.047, dernière=0.906, 4 époques.



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply r

### Lecture du résultat

La fixture de clipping utilise deux signes et distingue les pertes avec et sans clipping. Le gradient et la modification des poids sont mesurés ; PPO réalise 1600 mises à jour pour 400 batches.

## 6. GRPO : avantage relatif intra-groupe

Chaque prompt reçoit huit actions. L'écart-type est celui de l'échantillon (correction 1) et l'epsilon vaut 0,0001. Le callback de récompense peut être remplacé sans réécrire l'entraîneur.

In [6]:
Tensor GroupAdvantage(Tensor grouped,double epsilon=1e-4)
{
    if(grouped.shape.Length!=2||grouped.shape[1]<2||epsilon<=0||!double.IsFinite(epsilon))throw new ArgumentException("G >= 2 et epsilon positif requis");
    var mean=grouped.mean(new long[]{1},keepdim:true);
    var sd=grouped.std(new long[]{1},unbiased:true,keepdim:true);
    return (grouped-mean)/(sd+epsilon);
}
using(var groupScope=torch.NewDisposeScope())
{
    var groups=torch.tensor(new float[]{0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0}).reshape(2,8);
    var advantages=GroupAdvantage(groups);
    double sum=advantages[1].abs().sum().item<float>();
    double sd=groups.std(new long[]{1},unbiased:true,keepdim:true)[0].item<float>();
    var expected=torch.tensor(new float[]{0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0}).reshape(2,8);
    var expectedMean=torch.tensor(new float[]{0.375f,0}).reshape(2,1);
    var expectedSd=torch.tensor(new float[]{0.51754917f,0}).reshape(2,1);
    var oracle=(expected-expectedMean)/(expectedSd+1e-4);
    if(!double.IsFinite(sd)||Math.Abs(sd-0.51754917)>1e-5||!advantages.shape.SequenceEqual(new long[]{2,8})||advantages.isnan().any().item<bool>()||advantages.isinf().any().item<bool>()||(advantages-oracle).abs().max().item<float>()>1e-5||sum>1e-6||advantages[0].max().item<float>()<=0||advantages[0].min().item<float>()>=0)throw new Exception("Fixture de groupe incorrecte");
    Console.WriteLine($"Fixture : écart-type corrigé={sd:F6} ; avantage groupe constant={sum:F6} ; groupe contrasté de signes opposés.");
}
(List<double> history,torch.nn.Module<Tensor,Tensor> policy) TrainGrpo(int steps=400,int batch=32,int G=8,int seed=42,double epsilon=1e-4,Func<Tensor,Tensor,Tensor,Tensor> reward=null)
{
    if(G<2||epsilon<=0||!double.IsFinite(epsilon))throw new ArgumentException("G >= 2 et epsilon positif requis");
    reward ??= Reward;torch.manual_seed(seed);var rng=new Random(seed);
    var actor=Actor();using var optimizer=torch.optim.Adam(actor.parameters(),lr:0.005);
    var history=new List<double>();
    for(int step=0;step<steps;step++)
    {
        using var scope=torch.NewDisposeScope();
        var (a,b)=Batch(rng,batch,G);var at=torch.tensor(a);var bt=torch.tensor(b);var state=Encode(at,bt);
        Tensor action,oldLogp,r,advantage;
        using(torch.no_grad())
        {
            (action,oldLogp)=Sample(actor,state);r=reward(at,bt,action);
            if(!r.shape.SequenceEqual(new long[]{batch*G})||r.isnan().any().item<bool>()||r.isinf().any().item<bool>())throw new Exception("Reward invalide");
            advantage=GroupAdvantage(r.reshape(batch,G),epsilon).reshape(-1);
        }
        for(int epoch=0;epoch<4;epoch++)
        {
            using var epochScope=torch.NewDisposeScope();
            var lp=actor.forward(state).log_softmax(1);
            var logp=lp.gather(1,action.unsqueeze(1)).squeeze(1);
            var loss=ClippedLoss((logp-oldLogp).exp(),advantage);
            optimizer.zero_grad();loss.backward();optimizer.step();
        }
        history.Add(r.mean().item<float>());
    }
    return (history,actor);
}
int callbackCount=0;
var callbackHistory=TrainGrpo(steps:1,batch:2,G:2,reward:(a,b,act)=>{callbackCount++;return torch.ones_like(act).to_type(ScalarType.Float32);});
if(callbackCount!=1||callbackHistory.history.Count!=1||Math.Abs(callbackHistory.history[0]-1.0)>1e-6)throw new Exception("Callback de récompense non utilisé");
callbackHistory.policy.Dispose();
var grpoResult=TrainGrpo();var hGrpo=grpoResult.history;
Console.WriteLine($"GRPO : {hGrpo.Count} étapes, première récompense={hGrpo.First():F3}, dernière={hGrpo.Last():F3}, G=8, callback constant vérifié ({callbackCount} appel, reward 1).");

Fixture : écart-type corrigé=0.517549 ; avantage groupe constant=0.000000 ; groupe contrasté de signes opposés.


GRPO : 400 étapes, première récompense=0.074, dernière=1.000, G=8, callback constant vérifié (1 appel, reward 1).



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply r

### Lecture du résultat

Le groupe constant donne un avantage nul ; le groupe contrasté conserve des signes opposés. GRPO réalise 400 batches de 32 prompts, 256 actions par batch et 1600 mises à jour ; aucun critic n'est créé.

### Exercice 1 — Baseline médiane du groupe

Compléter l'avantage centré sur la médiane par groupe et réduit par l'écart-type corrigé. Pour un groupe pair, `torch.median` choisit la valeur centrale inférieure. Deux fixtures testent groupe constant et contrasté.

In [7]:
Tensor MedianAdvantage(Tensor rewards,double epsilon=1e-4)
{
    // TODO : mediane sur dim 1, ecart-type d'echantillon et normalisation.
    return null;
}
using(var scope=torch.NewDisposeScope())
{
    var fixture=torch.tensor(new float[]{0,0,1,1,2,2,2,2}).reshape(2,4);
    var answer=MedianAdvantage(fixture);
    if(answer is null)Console.WriteLine("Exercice 1 à compléter.");
    else
    {
        var expected=torch.tensor(new float[]{0,0,1,1,0,0,0,0}).reshape(2,4)/(fixture.std(new long[]{1},unbiased:true,keepdim:true)+1e-4);
        if(!answer.shape.SequenceEqual(new long[]{2,4})||answer.isnan().any().item<bool>()||answer.isinf().any().item<bool>()||(answer-expected).abs().max().item<float>()>1e-5)Console.WriteLine("Fixture médiane incorrecte.");
        else Console.WriteLine("Fixture médiane correcte ; comparer ensuite les courbes.");
    }
}

Exercice 1 à compléter.



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



## 7. Comparaison des courbes et des coûts

Les trois courbes comportent 400 mesures. Le lissage centré `w=10` utilise **11 points à l'intérieur**, moins aux bords. Les budgets diffèrent : 32/64/256 actions et 400/1600/1600 mises à jour pour REINFORCE/PPO/GRPO.

In [8]:
record SvgView(string Content);
Formatter.Register<SvgView>((v,w)=>w.Write(v.Content),"text/html");
string F(double x)=>x.ToString("0.###",CultureInfo.InvariantCulture);
double[] Smooth(IReadOnlyList<double> data,int w=10)
{
    return Enumerable.Range(0,data.Count).Select(i=>data.Skip(Math.Max(0,i-w/2)).Take(Math.Min(data.Count,i+w/2+1)-Math.Max(0,i-w/2)).Average()).ToArray();
}
string Chart(IReadOnlyList<double>[] histories,string[] names)
{
    var colors=new[]{"#b83a3a","#2e61a4","#19804a"};
    var sb=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='820' height='460'><rect width='820' height='460' fill='white'/><path d='M70 55 V360 H780' stroke='black' fill='none'/>");
    void Text(double x,double y,string text,string color="black")=>sb.Append($"<text x='{F(x)}' y='{F(y)}' fill='{color}' font-size='13'>{System.Security.SecurityElement.Escape(text)}</text>");
    Text(210,23,"Récompense moyenne : trois entraînements");Text(5,45,"récompense");Text(320,405,"itération (batch)");
    for(int i=0;i<=5;i++){double y=360-i*61;sb.Append($"<path d='M70 {F(y)} H780' stroke='#ddd'/>");Text(20,y+4,F(i/5.0));}
    for(int i=0;i<=4;i++){double x=70+i*177.5;sb.Append($"<path d='M{F(x)} 360 v5' stroke='black'/>");Text(x-10,380,F(i*100));}
    for(int series=0;series<histories.Length;series++)
    {
        var values=Smooth(histories[series]);sb.Append($"<polyline fill='none' stroke='{colors[series]}' stroke-width='2' points='");
        for(int i=0;i<values.Length;i++)sb.Append(F(70+i*710.0/399)+","+F(360-values[i]*305)+" ");
        sb.Append("'/>");Text(80+series*235,440,names[series],colors[series]);
    }
    sb.Append("</svg>");return sb.ToString();
}
var svg=Chart(new IReadOnlyList<double>[] {hReinforce,hPpo,hGrpo},new[]{"REINFORCE","PPO","GRPO"});
display(new SvgView(svg));
Console.WriteLine($"Courbes réelles : {hReinforce.Count}/{hPpo.Count}/{hGrpo.Count} points ; lissage centré 11 points intérieurs.");

Récompense moyenne : trois entraînements récompense itération (batch) 0 0.2 0.4 0.6 0.8 1 0 100 200 300 400 <polyline fill='none' stroke='#b83a3a' stroke-width='2' points='70,339.349 71.779,338.214 73.559,340.938 75.338,341.997 77.118,340.938 78.897,340.938 80.677,340.071 82.456,340.071 84.236,343.537 86.015,343.537 87.794,341.804 89.574,340.938 91.353,342.67 93.133,341.804 94.912,340.938 96.692,341.804 98.471,341.804 100.251,341.804 102.03,341.804 103.81,340.071 105.589,340.071 107.368,340.071 109.148,341.804 110.927,342.67 112.707,341.804 114.486,341.804 116.266,340.071 118.045,341.804 119.825,341.804 121.604,343.537 123.383,344.403 125.163,341.804 126.942,342.67 128.722,340.938 130.501,338.338 132.281,339.205 134.06,336.605 135.84,338.338 137.619,335.739 139.398,331.406 141.178,327.94 142.957,325.341 144.737,327.074 146.516,328.807 148.296,330.54 150.075,332.273 151.855,331.406 153.634,333.139 155.414,332.273 157.193,333.139 158.972,338.338 160.752,336.605 162.531,340.071 164.311,341.804 166.09,341.804 167.87,342.67 169.649,341.804 171.429,340.938 173.208,341.804 174.987,342.67 176.767,342.67 178.546,338.338 180.326,340.938 182.105,338.338 183.885,337.472 185.664,338.338 187.444,336.605 189.223,336.605 191.003,338.338 192.782,338.338 194.561,337.472 196.341,336.605 198.12,340.938 199.9,339.205 201.679,339.205 203.459,339.205 205.238,336.605 207.018,336.605 208.797,335.739 210.576,336.605 212.356,336.605 214.135,335.739 215.915,333.139 217.694,331.406 219.474,333.139 221.253,334.872 223.033,333.139 224.812,332.273 226.591,334.006 228.371,332.273 230.15,330.54 231.93,327.94 233.709,328.807 235.489,329.673 237.268,329.673 239.048,327.94 240.827,327.94 242.607,328.807 244.386,327.94 246.165,326.207 247.945,328.807 249.724,325.341 251.504,327.074 253.283,327.074 255.063,329.673 256.842,328.807 258.622,329.673 260.401,328.807 262.18,327.94 263.96,327.074 265.739,328.807 267.519,326.207 269.298,329.673 271.078,329.673 272.857,329.673 274.637,324.474 276.416,323.608 278.195,321.875 279.975,319.276 281.754,317.543 283.534,318.409 285.313,316.676 287.093,314.943 288.872,313.21 290.652,311.477 292.431,312.344 294.211,314.943 295.99,311.477 297.769,313.21 299.549,315.81 301.328,315.81 303.108,315.81 304.887,311.477 306.667,313.21 308.446,309.744 310.226,308.011 312.005,303.679 313.784,301.946 315.564,306.278 317.343,304.545 319.123,301.946 320.902,299.347 322.682,301.946 324.461,304.545 326.241,305.412 328.02,308.878 329.799,308.011 331.579,307.145 333.358,308.878 335.138,301.08 336.917,303.679 338.697,300.213 340.476,297.614 342.256,293.281 344.035,292.415 345.815,292.415 347.594,291.548 349.373,289.815 351.153,292.415 352.932,288.949 354.712,289.815 356.491,283.75 358.271,284.616 360.05,287.216 361.83,283.75 363.609,282.884 365.388,275.952 367.168,269.886 368.947,273.352 370.727,265.554 372.506,264.688 374.286,269.02 376.065,273.352 377.845,271.619 379.624,271.619 381.404,275.085 383.183,272.486 384.962,270.753 386.742,272.486 388.521,268.153 390.301,270.753 392.08,267.287 393.86,263.821 395.639,260.355 397.419,259.489 399.198,254.29 400.977,249.091 402.757,248.224 404.536,251.69 406.316,249.091 408.095,245.625 409.875,244.759 411.654,246.491 413.434,243.026 415.213,236.96 416.992,236.094 418.772,235.227 420.551,235.227 422.331,230.895 424.11,232.628 425.89,234.361 427.669,233.494 429.449,226.563 431.228,218.764 433.008,215.298 434.787,215.298 436.566,207.5 438.346,204.034 440.125,199.702 441.905,199.702 443.684,189.304 445.464,183.239 447.243,181.506 449.023,179.773 450.802,182.372 452.581,178.906 454.361,178.906 456.14,183.239 457.92,187.571 459.699,188.438 461.479,190.17 463.258,193.636 465.038,191.903 466.817,191.037 468.596,192.77 470.376,194.503 472.155,199.702 473.935,198.835 475.714,198.835 477.494,194.503 479.273,193.636 481.053,192.77 482.832,191.903 484.612,191.037 486.391,191.037 488.17,195.369 489.95,191.037 491.729,185.838 493.509,180.639 495.288,179.773 497.068,180.639 498.847,178.04 500.627,171.974 502.406,172.841 504

Courbes réelles : 400/400/400 points ; lissage centré 11 points intérieurs.


### Lecture du graphique

Dans cette exécution les dernières récompenses sont 0,656 pour REINFORCE, 0,906 pour PPO et 1,000 pour GRPO. Elles proviennent des batches d'entraînement et ne constituent ni une évaluation hors échantillon ni un classement universel.

### Exercice 2 — Taille du groupe G

Relancer l'entraînement à G=2, 4, 8 et 16 pendant 200 étapes, relever et tracer la récompense finale en fonction de G, puis comparer les coûts. Deux fonctions déterministes contrôlent les appels et les clés ; aucune monotonie n'est imposée.

In [9]:
Dictionary<int,double> EvaluerGroupes(Func<int,double> entrainer)
{
    // TODO : appeler entrainer pour G = 2, 4, 8 et 16.
    return null;
}
var seen1=new List<int>();var f1=EvaluerGroupes(g=>{seen1.Add(g);return g/20.0;});
if(f1 is null)Console.WriteLine("Exercice 2 à compléter.");
else
{
    var seen2=new List<int>();var f2=EvaluerGroupes(g=>{seen2.Add(g);return 1.0/g;});int[] sizes={2,4,8,16};
    bool ok=f2!=null&&f1.Keys.OrderBy(x=>x).SequenceEqual(sizes)&&f2.Keys.OrderBy(x=>x).SequenceEqual(sizes)&&seen1.OrderBy(x=>x).SequenceEqual(sizes)&&seen2.OrderBy(x=>x).SequenceEqual(sizes)&&sizes.All(g=>double.IsFinite(f1[g])&&double.IsFinite(f2[g])&&Math.Abs(f1[g]-g/20.0)<1e-9&&Math.Abs(f2[g]-1.0/g)<1e-9);
    if(!ok)Console.WriteLine("Fixtures incorrectes.");
    else
    {
        var result=EvaluerGroupes(g=>{var trained=TrainGrpo(steps:200,G:g);double value=trained.history.Last();trained.policy.Dispose();return value;});
        if(result==null||!result.Keys.OrderBy(x=>x).SequenceEqual(sizes)||result.Values.Any(x=>!double.IsFinite(x)||x<0||x>1))Console.WriteLine("Résultat invalide.");
        else
        {
            foreach(var x in result)Console.WriteLine($"G={x.Key} : {x.Value:F3}");
            var graph=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='580' height='380'><rect width='580' height='380' fill='white'/><path d='M70 40 V300 H540' fill='none' stroke='black'/>");
            for(int tick=0;tick<=5;tick++)
            {
                double y=300-tick*52;
                graph.Append($"<path d='M65 {F(y)} H540' stroke='#ddd'/><text x='20' y='{F(y+4)}' font-size='13'>{F(tick/5.0)}</text>");
            }
            graph.Append("<text x='180' y='20' font-size='15'>Récompense finale selon G</text><text x='5' y='35' font-size='13'>reward</text><text x='260' y='355' font-size='13'>taille du groupe G</text><polyline fill='none' stroke='#1763a5' stroke-width='2' points='");
            for(int i=0;i<sizes.Length;i++)graph.Append(F(90+i*140)+","+F(300-result[sizes[i]]*260)+" ");
            graph.Append("'/>");
            for(int i=0;i<sizes.Length;i++)
            {
                double x=90+i*140,y=300-result[sizes[i]]*260;
                graph.Append($"<circle cx='{F(x)}' cy='{F(y)}' r='4' fill='#1763a5'/><text x='{F(x-5)}' y='323' font-size='13'>{sizes[i]}</text>");
            }
            graph.Append("</svg>");display(new SvgView(graph.ToString()));
            Console.WriteLine("Courbe des récompenses finales par taille de groupe.");
        }
    }
}

Exercice 2 à compléter.


### Contrôle des modèles entraînés

Les politiques retournées et le critic restent utilisables après la fin de leurs optimiseurs, puis sont libérés explicitement.

In [10]:
using(var scope=torch.NewDisposeScope())
{
    var at=torch.tensor(new long[]{2});var bt=torch.tensor(new long[]{6});var state=Encode(at,bt);
    var a=reinforceResult.policy.forward(state);var b=ppoResult.policy.forward(state);var c=grpoResult.policy.forward(state);var value=ppoResult.critic.forward(state);
    if(!a.shape.SequenceEqual(new long[]{1,16})||!b.shape.SequenceEqual(new long[]{1,16})||!c.shape.SequenceEqual(new long[]{1,16})||!value.shape.SequenceEqual(new long[]{1,1})||(a.isnan().any().item<bool>()||a.isinf().any().item<bool>())||(b.isnan().any().item<bool>()||b.isinf().any().item<bool>())||(c.isnan().any().item<bool>()||c.isinf().any().item<bool>())||(value.isnan().any().item<bool>()||value.isinf().any().item<bool>()))throw new Exception("Modèle retourné inutilisable");
    Console.WriteLine("Politiques entraînées : trois sorties (1,16) et critic (1,1), valeurs finies.");
}
reinforceResult.policy.Dispose();ppoResult.policy.Dispose();ppoResult.critic.Dispose();grpoResult.policy.Dispose();actorCount.Dispose();criticCount.Dispose();solver.Dispose();ctx.Dispose();
Console.WriteLine("Modèles et solveur libérés après utilisation.");

Politiques entraînées : trois sorties (1,16) et critic (1,1), valeurs finies.


Modèles et solveur libérés après utilisation.



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



### Mise en regard : technique séduisante et coût mesuré

[JohnEnev, série « modern-llm » Part 2](https://johne.substack.com/p/modern-llm-part-2-modernizing-the-architecture) (8 juillet 2026), rapporte sur son modèle de 315 M paramètres une baisse de throughput de 41 % à deux streams et 49 % à quatre, ainsi qu'un coût d'entraînement de 93 à 155 puis 175 dollars pour mHC sans gain de validation observé. Il rapporte avoir écarté cette variante. Ce sont ses résultats, non une expérience reproduite ici ; une autre échelle ou infrastructure pourrait différer. Le lien avec ce carnet est méthodologique : mesurer coût et qualité avant de conclure, sans assimiler mHC et GRPO.

### Exercice 3 — Récompense dense

Compléter $1-|action^2+a-b|/230$ borné entre 0 et 1. Après contrôle de fixtures distinctes, le callback est transmis à `TrainGrpo` pour une courte expérience : prédire puis mesurer son effet.

In [11]:
Tensor DenseReward(Tensor a,Tensor b,Tensor action)
{
    // TODO : calculer le résidu, normaliser par 230 et borner à [0,1].
    return null;
}
using(var scope=torch.NewDisposeScope())
{
    var aa=torch.tensor(new long[]{2,1,5,0});var bb=torch.tensor(new long[]{6,5,230,0});var act=torch.tensor(new long[]{2,3,0,15});
    var answer=DenseReward(aa,bb,act);
    if(answer is null)Console.WriteLine("Exercice 3 à compléter.");
    else
    {
        var expected=torch.tensor(new float[]{1f,1f-5f/230f,1f-225f/230f,1f-225f/230f});
        bool ok=answer.shape.SequenceEqual(new long[]{4})&&!answer.isnan().any().item<bool>()&&!answer.isinf().any().item<bool>()&&(answer-expected).abs().max().item<float>()<1e-5;
        if(!ok)Console.WriteLine("Fixture de récompense dense incorrecte.");
        else
        {
            int calls=0;var dense=TrainGrpo(steps:20,reward:(a,b,x)=>{calls++;return DenseReward(a,b,x);});
            if(calls!=20||dense.history.Any(x=>!double.IsFinite(x)))Console.WriteLine("Callback dense incorrect.");
            else Console.WriteLine($"Récompense dense : {calls} appels, dernière moyenne {dense.history.Last():F3}.");
            dense.policy.Dispose();
        }
    }
}

Exercice 3 à compléter.



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply r

## Exercices

- [Médiane](#exercice-1--baseline-médiane-du-groupe) : compléter l'avantage, vérifier groupes constant et contrasté.
- [Taille du groupe](#exercice-2--taille-du-groupe-g) : mesurer G=2, 4, 8, 16 pendant 200 étapes.
- [Reward dense](#exercice-3--récompense-dense) : normaliser le résidu par 230 et utiliser le callback d'entraînement.

## À retenir

La fast-path a été confrontée à 2400 vérifications Z3. GRPO utilise ici un baseline empirique de groupe ; les 96 prompts finis et l'absence de holdout limitent l'interprétation. Supprimer un critic de 15 297 paramètres dans ce toy ne quantifie pas la mémoire totale d'un LLM.

## Vérifiez votre compréhension

<details><summary>Pourquoi un groupe à récompenses identiques ne fournit-il pas de signal ?</summary>La soustraction de la moyenne donne zéro pour chaque membre.</details>

<details><summary>Les 400 étapes ont-elles le même coût pour les trois méthodes ?</summary>Non : elles évaluent respectivement 32, 64 et 256 actions par étape et effectuent une ou quatre mises à jour.</details>

## Pour aller plus loin

Évaluer plusieurs seeds et un jeu séparé, puis comparer à [PT-04](PT_04_grpo_deepseek_r1.ipynb), où les séquences et la régularisation KL interviennent.